### Applying K Means Clusteting algorithm to cluster satellites into 'shells'
- Since our dataset is completely ready for ML models, we will now group similar satellites together
using k means clustering unsupervised model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [ ]:
df = pd.read_csv('../data/model_ready/satellites_final.csv')

In [ ]:
df.head()

#### Import necessary libraries

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
# Create a model (we will use a random value for K), with a random state as 101
model = KMeans(n_clusters=5, random_state=101)

In [ ]:
# Train and test the model
cluster_labels = model.fit_predict(df)

In [ ]:
# predicted cluster label for each satellite
cluster_labels

In [ ]:
# Make a seperate column
df['CLUSTER'] = cluster_labels

In [ ]:
# Let's find the occurence of each cluster
df['CLUSTER'].value_counts().sort_values()

---
#### The question is, how do we even interprete this, as we dont even know what that 0, 1 or any other cluster actually is..

In [ ]:
df.corr(numeric_only=True)['CLUSTER'].sort_values()

- Let's plot the above thing

In [ ]:
plt.figure(figsize=(12, 6))
df.corr(numeric_only=True)['CLUSTER'].sort_values()[:-1].plot(kind='bar')

plt.xticks(rotation=90, ha = 'right', fontsize = 12)
plt.tight_layout();

#### Interpreting the results for K = 5

- Let's consider the features having the most association with the cluster (both positive and negative)
- The featues most correlated are :
- ARG_OF_.., MEAN_MOTION_DOT, INCLINATION, AGE_SINCE_LAUNCH, REV_AT_EPOCH, MEAN_ANOMALY

In [ ]:
most_corr_features = ['ARG_OF_PERICENTER', 'MEAN_MOTION_DOT', 'INCLINATION', 
                     'AGE_SINCE_LAUNCH', 'REV_AT_EPOCH', 'MEAN_ANOMALY'] 

In [ ]:
grouped = df.groupby('CLUSTER')[most_corr_features].agg(['mean'])
grouped

In [ ]:
grouped.T.plot(kind='bar', figsize=(12,6)) 
plt.title("Mean feature values by cluster (log scale)")
plt.ylabel("Mean value (log scale)")
plt.xlabel("Feature")
plt.legend(loc=[1.1, 0.6])
plt.show()


- Physical meaning: 

| Cluster | Main Characteristic                             | Possible Satellite Type                     |
| ------- | ----------------------------------------------- | ------------------------------------------- |
| 0       | Newer, low-inclination, similar orientation     | Recently launched LEO constellations        |
| 1       | Mid-age, polar inclination                      | Earth-observation / reconnaissance          |
| 2       | Average across features                         | Transitional group                          |
| 3       | Old, low-moderate inclination, many revolutions | Aging operational satellites                |
| 4       | Very old, high-inclination                      | Polar / scientific or navigation satellites |


---
### Choosing optimal K value for clustering

In [ ]:
ssd = []

# loop through k = 2 to 10 (basic estimation)
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=101)

    model.fit_predict(df) # Train the model
    ssd.append(model.inertia_) # append the SSD / inertia to the list

In [ ]:
ssd # summed up ssd for each k value

In [ ]:
# Let's plot this
plt.plot(range(2, 11), ssd, 'o--')

- Let's find the difference between the SSDs

In [ ]:
pd.Series(ssd).diff()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Your SSD values for k=2 to 10
K_range = range(2, 11)
# replace with your real SSD values

# Convert to Series for diff calculation
ssd_series = pd.Series(ssd, index=K_range)
ssd_diff = ssd_series.diff()

# Plot SSD vs K
plt.figure(figsize=(10,5))
plt.plot(K_range, ssd, 'o-', label='SSD (Inertia)')
plt.title('Elbow Method for KMeans')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Sum of squared distances (SSD)')
plt.grid(True)

# Optional: show the difference as a bar
plt.twinx()
plt.bar(K_range, ssd_diff, alpha=0.3, color='orange', label='SSD diff')
plt.ylabel('Difference in SSD')
plt.legend(loc='upper right')

plt.show()


---
### KMeans Clustering Interpretation

We applied KMeans clustering to our scaled satellite dataset and used the **Elbow Method** to determine the optimal number of clusters.  

**Elbow Analysis:**  
- The plot of **Sum of Squared Distances (SSD) vs. number of clusters (k)** shows that SSD decreases sharply at first, then the rate of decrease slows down.  
- The **difference in SSD** (`.diff()`) highlights how much each additional cluster improves clustering.  
- Observing the plot, the “elbow” appears around **k = 5**, where adding more clusters provides diminishing returns.  

**Cluster Summary (k = 5):**  
1. **Cluster 0:**  
   - Low inclination (orbits close to equator)  
   - ARG_OF_PERICENTER above average → closest approach happens later along orbit  
   - Relatively new satellites  

2. **Cluster 1:**  
   - Moderate-to-high inclination → satellites tilted more from equator  
   - Average orbital orientation and age  

3. **Cluster 2:**  
   - Features close to dataset average → “typical” satellites  
   - No extreme orbital characteristics  

4. **Cluster 3:**  
   - High age and number of revolutions → older satellites  
   - Moderate inclination and ARG_OF_PERICENTER  

5. **Cluster 4:**  
   - Oldest satellites with many revolutions  
   - High inclination → likely polar or near-polar orbits  
   - Moderate ARG_OF_PERICENTER  

**Conclusion:**  
- **k = 5 is a reasonable choice**: it balances capturing the main patterns in satellite orbital characteristics without overcomplicating the model with too many clusters.  
- Each cluster represents **distinct satellite groups** based on a combination of **inclination, orbital orientation (ARG_OF_PERICENTER), age, and revolutions**, which are the main differentiating factors in this dataset.
